# Giacomini-White Test and Forecast Encompassing

This notebook covers two complementary forecast evaluation approaches:

1. **Giacomini-White (2006) test** — Tests *conditional* predictive ability using instrumental variables. Unlike the DM test (which tests *unconditional* equal accuracy), GW detects cases where one model is better in some states of the economy.

2. **Mincer-Zarnowitz regression** — Tests forecast *efficiency* (unbiasedness + calibration) via $y_t = \alpha + \beta \hat{y}_t + \varepsilon_t$.

3. **Forecast Encompassing** (Harvey, Leybourne & Newbold, 1998) — Tests whether one forecast contains all useful information from another.

**References:**
- Giacomini, R. & White, H. (2006). "Tests of Conditional Predictive Ability." *Econometrica*, 74(6), 1545-1578.
- Mincer, J.A. & Zarnowitz, V. (1969). "The Evaluation of Economic Forecasts." *NBER*.
- Harvey, D., Leybourne, S. & Newbold, P. (1998). "Tests for Forecast Encompassing." *JBES*, 16(2), 254-259.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

from forecastbox.evaluation import (
    giacomini_white, GWResult,
    diebold_mariano, DMResult,
    mincer_zarnowitz, MZResult,
    encompassing_test, EncompassingResult,
    model_confidence_set, MCSResult,
)
from utils.helpers import load_inflation_forecasts, load_m4_sample

## 1. Giacomini-White (2006) Test

The DM test checks whether $E[d_t] = 0$ (unconditional). But what if model A is better during recessions and model B during expansions? On average they may look equal, but conditionally they differ.

The **GW test** uses instruments $h_t$ to test:

$$H_0: E[h_t \cdot d_t] = 0$$

Default instruments are $h_t = [1, d_{t-1}]$ (a constant and the lagged loss differential), which capture persistence in relative forecast performance.

In [ ]:
# Load data
df = load_inflation_forecasts()
actual = df["actual"].values
fc_arima = df["fc_arima"].values
fc_ets = df["fc_ets"].values
fc_var = df["fc_var"].values
fc_naive = df["fc_naive"].values
fc_drift = df["fc_drift"].values

model_names = ["ARIMA", "ETS", "VAR", "Naive", "Drift"]
forecasts_list = [fc_arima, fc_ets, fc_var, fc_naive, fc_drift]

# GW test: ARIMA vs ETS with default instruments [1, d_{t-1}]
gw_result = giacomini_white(actual, fc_arima, fc_ets, h=1, loss="mse")

print("Giacomini-White Test: ARIMA vs ETS")
print("=" * 55)
print(f"GW statistic (chi2): {gw_result.statistic:.4f}")
print(f"p-value:             {gw_result.pvalue:.4f}")
print(f"Degrees of freedom:  {gw_result.df}")
print(f"Instruments:         {gw_result.instruments_used}")
print(f"\n{gw_result.conclusion()}")

# GW test for all pairs
print("\n\nGW Pairwise Tests (default instruments)")
print("-" * 55)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        res = giacomini_white(actual, forecasts_list[i], forecasts_list[j], h=1, loss="mse")
        sig = "*" if res.pvalue < 0.05 else ""
        print(f"  {model_names[i]:>8} vs {model_names[j]:<8}: chi2={res.statistic:>7.3f}, p={res.pvalue:.4f} {sig}")

## 2. GW vs DM

Key differences between the Giacomini-White and Diebold-Mariano tests:

| Feature | DM Test | GW Test |
|---------|---------|---------|
| **Hypothesis** | Unconditional: $E[d_t] = 0$ | Conditional: $E[h_t \cdot d_t] = 0$ |
| **Power** | Against constant differences | Against state-dependent differences |
| **Distribution** | t or Normal | $\chi^2(q)$ where $q$ = # instruments |
| **Use case** | "Which model is better on average?" | "Is one model conditionally better?" |

When the true predictive ability difference is constant over time, DM and GW should give similar conclusions. GW has more power when the difference varies with the state.

In [ ]:
# Compare DM vs GW results for the same pairs
print("DM vs GW Comparison")
print("=" * 70)
print(f"{'Pair':<20} {'DM stat':>10} {'DM p':>8} {'GW stat':>10} {'GW p':>8} {'Agree?':>8}")
print("-" * 70)

pairs = [
    ("ARIMA vs ETS", fc_arima, fc_ets),
    ("ARIMA vs Naive", fc_arima, fc_naive),
    ("VAR vs ETS", fc_var, fc_ets),
    ("ETS vs Drift", fc_ets, fc_drift),
    ("VAR vs Naive", fc_var, fc_naive),
]

for pair_name, fc1, fc2 in pairs:
    dm = diebold_mariano(actual, fc1, fc2, h=1, loss="mse")
    gw = giacomini_white(actual, fc1, fc2, h=1, loss="mse")
    
    dm_reject = dm.pvalue < 0.05
    gw_reject = gw.pvalue < 0.05
    agree = "Yes" if dm_reject == gw_reject else "NO"
    
    print(f"{pair_name:<20} {dm.statistic:>10.3f} {dm.pvalue:>8.4f} "
          f"{gw.statistic:>10.3f} {gw.pvalue:>8.4f} {agree:>8}")

print("\nNote: When DM and GW disagree, GW may be detecting conditional differences")
print("that are masked in the unconditional DM test (or vice versa).")

## 3. Mincer-Zarnowitz Regression

The Mincer-Zarnowitz (1969) regression tests forecast **efficiency** (also called rationality):

$$y_t = \alpha + \beta \hat{y}_t + \varepsilon_t$$

An efficient forecast satisfies $H_0: \alpha = 0, \beta = 1$ jointly. This means:
- $\alpha = 0$: no systematic bias
- $\beta = 1$: the forecast correctly scales the actual variation

Failure indicates the forecast is either biased ($\alpha \neq 0$) or miscalibrated ($\beta \neq 1$).

In [ ]:
# Mincer-Zarnowitz regression for each model
print("Mincer-Zarnowitz Efficiency Tests")
print("=" * 75)
print(f"{'Model':<10} {'alpha':>8} {'beta':>8} {'R2':>8} {'F-stat':>10} {'p-value':>10} {'Efficient?':>12}")
print("-" * 75)

mz_results = {}
for name, fc in zip(model_names, forecasts_list):
    mz = mincer_zarnowitz(actual, fc)
    mz_results[name] = mz
    eff = "Yes" if mz.is_efficient() else "No"
    print(f"{name:<10} {mz.alpha:>8.4f} {mz.beta:>8.4f} {mz.r_squared:>8.4f} "
          f"{mz.f_statistic:>10.3f} {mz.pvalue:>10.4f} {eff:>12}")

# Detailed output for the best model
print("\n\nDetailed MZ Results for ARIMA:")
print(mz_results["ARIMA"].summary())

# Scatter plot: actual vs forecast for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, fc) in zip(axes, [("ARIMA", fc_arima), ("ETS", fc_ets), ("VAR", fc_var)]):
    mz = mz_results[name]
    ax.scatter(fc, actual, alpha=0.5, s=15)
    # Plot regression line
    x_range = np.linspace(fc.min(), fc.max(), 100)
    ax.plot(x_range, mz.alpha + mz.beta * x_range, "r-", label=f"MZ: a={mz.alpha:.3f}, b={mz.beta:.3f}")
    ax.plot(x_range, x_range, "k--", alpha=0.5, label="45-degree line")
    ax.set_xlabel("Forecast")
    ax.set_ylabel("Actual")
    ax.set_title(f"{name} (p={mz.pvalue:.3f})")
    ax.legend(fontsize=8)

plt.suptitle("Mincer-Zarnowitz Regressions", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Forecast Encompassing Test

The encompassing test (Harvey, Leybourne & Newbold, 1998) asks: **does forecast 1 contain all the useful information in forecast 2?**

The regression:

$$y_t - \hat{y}_{2,t} = \lambda (\hat{y}_{1,t} - \hat{y}_{2,t}) + \varepsilon_t$$

- $\lambda = 0$: forecast 2 encompasses forecast 1 (f1 adds nothing)
- $\lambda = 1$: forecast 1 encompasses forecast 2 (f2 adds nothing)
- $0 < \lambda < 1$: neither encompasses the other — both have unique information, suggesting a combination could improve

In [ ]:
# Forecast encompassing tests
print("Forecast Encompassing Tests")
print("=" * 70)
print(f"{'f1':>10} {'f2':>10} {'lambda':>8} {'t-stat':>8} {'p-val':>8} {'Conclusion':<30}")
print("-" * 70)

for i in range(len(model_names)):
    for j in range(len(model_names)):
        if i == j:
            continue
        enc = encompassing_test(actual, forecasts_list[i], forecasts_list[j])
        
        if enc.f1_encompasses_f2:
            conclusion = f"{model_names[i]} encompasses {model_names[j]}"
        elif enc.f2_encompasses_f1:
            conclusion = f"{model_names[j]} encompasses {model_names[i]}"
        elif enc.neither_encompasses:
            conclusion = "Neither encompasses"
        else:
            conclusion = "Inconclusive"
        
        print(f"{model_names[i]:>10} {model_names[j]:>10} {enc.lambda_hat:>8.4f} "
              f"{enc.statistic:>8.3f} {enc.pvalue:>8.4f} {conclusion:<30}")

# Detailed result for one pair
print("\n\nDetailed: ARIMA vs ETS")
enc_detail = encompassing_test(actual, fc_arima, fc_ets)
print(enc_detail.summary())

## 5. Complete Evaluation Report

Bringing together all four evaluation tools — DM, MCS, Mincer-Zarnowitz, and encompassing — into a single comprehensive forecast evaluation report.

In [ ]:
# Complete Evaluation Report
forecasts_dict = dict(zip(model_names, forecasts_list))

print("=" * 70)
print("           COMPLETE FORECAST EVALUATION REPORT")
print("=" * 70)

# 1. Point accuracy (MSE)
print("\n1. POINT ACCURACY (MSE)")
print("-" * 40)
for name, fc in zip(model_names, forecasts_list):
    mse = np.mean((actual - fc) ** 2)
    mae = np.mean(np.abs(actual - fc))
    print(f"  {name:<10}: MSE={mse:.6f}, MAE={mae:.6f}")

# 2. Mincer-Zarnowitz efficiency
print("\n2. FORECAST EFFICIENCY (Mincer-Zarnowitz)")
print("-" * 40)
for name, fc in zip(model_names, forecasts_list):
    mz = mincer_zarnowitz(actual, fc)
    eff = "Efficient" if mz.is_efficient() else "NOT efficient"
    print(f"  {name:<10}: alpha={mz.alpha:.4f}, beta={mz.beta:.4f}, p={mz.pvalue:.4f} -> {eff}")

# 3. Model Confidence Set
print("\n3. MODEL CONFIDENCE SET (alpha=0.10)")
print("-" * 40)
mcs = model_confidence_set(actual, forecasts_dict, alpha=0.10, statistic="range", n_boot=5000, seed=42)
print(f"  Included: {mcs.included_models}")
print(f"  Excluded: {mcs.excluded_models}")
print(f"  Elimination order: {mcs.elimination_order}")

# 4. Pairwise DM tests (significant pairs)
print("\n4. SIGNIFICANT PAIRWISE DIFFERENCES (DM, p<0.05)")
print("-" * 40)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        dm = diebold_mariano(actual, forecasts_list[i], forecasts_list[j])
        if dm.pvalue < 0.05:
            better = model_names[i] if dm.mean_loss_diff < 0 else model_names[j]
            print(f"  {model_names[i]} vs {model_names[j]}: p={dm.pvalue:.4f} ({better} is better)")

# 5. Encompassing summary
print("\n5. ENCOMPASSING RESULTS")
print("-" * 40)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        enc = encompassing_test(actual, forecasts_list[i], forecasts_list[j])
        if enc.neither_encompasses:
            print(f"  {model_names[i]} & {model_names[j]}: neither encompasses -> combine!")
        elif enc.f1_encompasses_f2:
            print(f"  {model_names[i]} encompasses {model_names[j]}")
        elif enc.f2_encompasses_f1:
            print(f"  {model_names[j]} encompasses {model_names[i]}")

# 6. GW conditional differences
print("\n6. CONDITIONAL PREDICTIVE ABILITY (Giacomini-White, p<0.05)")
print("-" * 40)
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        gw = giacomini_white(actual, forecasts_list[i], forecasts_list[j])
        if gw.pvalue < 0.05:
            print(f"  {model_names[i]} vs {model_names[j]}: chi2={gw.statistic:.3f}, p={gw.pvalue:.4f}")

print("\n" + "=" * 70)

## Exercise 1: GW test with different instrument sets

Try using different instruments in the GW test — for example, add lagged actual values or squared lagged loss differentials. Do the results change?

In [ ]:
# TODO: Exercise 1
# Hint: Build custom instrument matrices and pass to giacomini_white()
# 
# # Compute loss differential for instruments
# d = (actual - fc_arima)**2 - (actual - fc_ets)**2
# 
# # Instrument set 1: [1, d_{t-1}] (default)
# # Instrument set 2: [1, d_{t-1}, actual_{t-1}]
# # Instrument set 3: [1, d_{t-1}, d_{t-1}^2]
#
# # Example: instruments with lagged actual
# h_mat = np.column_stack([np.ones(len(actual)-1), d[:-1], actual[:-1]])
# gw_custom = giacomini_white(actual[1:], fc_arima[1:], fc_ets[1:], instruments=h_mat)
# print(gw_custom.conclusion())

## Exercise 2: Build a complete evaluation for M4 sample

Using the `m4_sample.csv` data, run a complete evaluation (DM pairwise, MCS, MZ efficiency, encompassing) for each of the 6 M4 series. Summarize the results across all series.

In [ ]:
# TODO: Exercise 2
# Hint:
# m4 = load_m4_sample()
# for series_id in m4["series_id"].unique():
#     subset = m4[m4["series_id"] == series_id]
#     actual_s = subset["actual"].values
#     fc_dict = {
#         "Model1": subset["fc_model1"].values,
#         "Model2": subset["fc_model2"].values,
#         "Model3": subset["fc_model3"].values,
#     }
#     # DM pairwise
#     # MCS
#     # MZ for each model
#     # Encompassing for each pair
#     ...